# 동아일보 기사 크롤링

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException

import pandas as pd
import time
import re
import csv
import ast

In [16]:
# Headless 모드 설정
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')

# 드라이버 경로 설정
service = Service(executable_path='../chromedriver-mac-arm64/chromedriver')
driver = webdriver.Chrome(service=service, options=options)

# 첫 페이지 열기
driver.get("https://www.donga.com/news/Economy/RE")

# 로딩 대기
wait = WebDriverWait(driver, 5)
wait.until(EC.presence_of_element_located((
    By.CSS_SELECTOR,
    '#contents > div > div > div.divide_area > section > ul > li:nth-child(1) > article > div > h4 > a'
)))

<selenium.webdriver.remote.webelement.WebElement (session="380936ba2c61f5059a1447ac4d960640", element="f.390B1371C6DA40F4AB4294E915181389.d.BAAA3109EF8C167E5981738D0A5DC66F.e.66")>

In [17]:
# 기준 날짜 설정 (2020년 9월 1일)
cutoff_date = 20200901

# 최대 페이지 수
MAX_PAGE = 1
article_links = set()

for page in range(1, MAX_PAGE + 1):
    offset = (page - 1) * 20 + 1
    url = f"https://www.donga.com/news/Economy/RE?p={offset}&prod=news&ymd=&m="

    try:
        driver.get(url)
        print(f"[INFO] Visiting page {page} -> {url}")
        time.sleep(2)

        links = driver.find_elements(By.CSS_SELECTOR, "a")
        for link in links:
            href = link.get_attribute("href")
            if href and "https://www.donga.com/news/Economy/article/all/" in href:
                # 날짜 추출 (정규표현식 사용)
                match = re.search(r'/all/(\d{8})/', href)
                if match:
                    article_date = int(match.group(1))
                    if article_date >= cutoff_date:
                        article_links.add(href)
    except Exception as e:
        print(f"[ERROR] Failed to process page {page}: {e}")

# 결과 출력
print(f"\n✅ 수집된 기사 링크 수 (2020년 9월 1일 이후): {len(article_links)}")
for link in sorted(article_links):
    print(link)


[INFO] Visiting page 1 -> https://www.donga.com/news/Economy/RE?p=1&prod=news&ymd=&m=

✅ 수집된 기사 링크 수 (2020년 9월 1일 이후): 21
https://www.donga.com/news/Economy/article/all/20250612/131788309/2
https://www.donga.com/news/Economy/article/all/20250612/131788317/2
https://www.donga.com/news/Economy/article/all/20250612/131788320/2
https://www.donga.com/news/Economy/article/all/20250612/131788323/2
https://www.donga.com/news/Economy/article/all/20250612/131788326/2
https://www.donga.com/news/Economy/article/all/20250612/131788329/2
https://www.donga.com/news/Economy/article/all/20250612/131796517/2
https://www.donga.com/news/Economy/article/all/20250612/131796519/2
https://www.donga.com/news/Economy/article/all/20250612/131796521/2
https://www.donga.com/news/Economy/article/all/20250612/131796674/2
https://www.donga.com/news/Economy/article/all/20250613/131797676/2
https://www.donga.com/news/Economy/article/all/20250613/131798735/1
https://www.donga.com/news/Economy/article/all/20250613/131799

In [18]:
driver.quit()


In [19]:
article_links = list(article_links)

In [21]:
# Headless 모드 설정
options = Options()
options.add_argument('--headless')
options.add_argument('--no-sandbox')
options.add_argument('--disable-dev-shm-usage')


# 드라이버 경로 설정
service = Service(executable_path='../chromedriver-mac-arm64/chromedriver')
driver = webdriver.Chrome(service=service, options=options)

article = {}

for i, url in enumerate(article_links):
    print(f"\n[INFO] ({i+1}/{len(article_links)}) URL 접속 중: {url}")
    try:
        driver.get(url)
        time.sleep(2)  # JS 렌더링 대기

        try:
            # section.news_view 요소 찾기
            section = driver.find_element(By.CSS_SELECTOR, 'section.news_view')

            # 광고, script, iframe 등 불필요 태그 제거
            driver.execute_script("""
                const section = arguments[0];
                const tags = section.querySelectorAll('script, style, iframe, div.a1, div.view_ad06, div.view_m_adA, div.view_m_adB');
                tags.forEach(tag => tag.remove());
            """, section)

            # innerText로 텍스트 추출
            full_text = section.get_attribute('innerText').strip()

            if not full_text:
                full_text = "본문 없음 (빈 본문)"
                print(f"[WARNING] 본문이 비어 있음")
            else:
                print(f"[DEBUG] 본문 추출 성공 ({len(full_text)}자)")

        except Exception as e:
            full_text = '본문 없음'
            print(f"[WARNING] 본문 추출 실패: {e}")

        # 저장
        article[url] = {
            'content': full_text
        }

    except Exception as e:
        print(f"[ERROR] URL 접근 실패: {url} | 에러: {e}")
        article[url] = {
            'content': '접근 실패'
        }

# 드라이버 종료
driver.quit()
print("\n[INFO] 크롤링 완료.")


[INFO] (1/21) URL 접속 중: https://www.donga.com/news/Economy/article/all/20250613/131799238/1
[DEBUG] 본문 추출 성공 (1486자)

[INFO] (2/21) URL 접속 중: https://www.donga.com/news/Economy/article/all/20250612/131796674/2
[DEBUG] 본문 추출 성공 (883자)

[INFO] (3/21) URL 접속 중: https://www.donga.com/news/Economy/article/all/20250612/131788309/2
[DEBUG] 본문 추출 성공 (1817자)

[INFO] (4/21) URL 접속 중: https://www.donga.com/news/Economy/article/all/20250613/131803400/1
[DEBUG] 본문 추출 성공 (1489자)

[INFO] (5/21) URL 접속 중: https://www.donga.com/news/Economy/article/all/20250613/131797676/2
[DEBUG] 본문 추출 성공 (2059자)

[INFO] (6/21) URL 접속 중: https://www.donga.com/news/Economy/article/all/20250612/131788326/2
[DEBUG] 본문 추출 성공 (1020자)

[INFO] (7/21) URL 접속 중: https://www.donga.com/news/Economy/article/all/20250615/131807019/1
[DEBUG] 본문 추출 성공 (2564자)

[INFO] (8/21) URL 접속 중: https://www.donga.com/news/Economy/article/all/20250612/131796519/2
[DEBUG] 본문 추출 성공 (778자)

[INFO] (9/21) URL 접속 중: https://www.donga.com/news/Econom

In [22]:
# 크롤링 내용 csv로 저장

with open('../data/raw/news/dong_a_ilbo.csv', 'w', newline="", encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['URL', 'content'])

    for url, content in article.items():
        writer.writerow([url, content])
    

In [23]:
driver.quit()

# 뉴스기사 전처리

## 결측치 처리 및 월별로 정렬

In [89]:
dong_a = pd.read_csv("../data/raw/news/dong_a_ilbo.csv")

In [90]:
dong_a.head()

,URL,content
0,https://www.donga.com/news/Economy/article/all...,{'content': '수도권 전용 59㎡ 분양가 7억…1년새 13.5% 올라\n청...
1,https://www.donga.com/news/Economy/article/all...,{'content': '강남 3구 이어 성동-마포 ‘풍선효과’\n정부 부동산TF “...
2,https://www.donga.com/news/Economy/article/all...,{'content': '롯데캐슬 르웨스트\n풀 퍼니시드 적용… 즉시 입주 가능\n가...
3,https://www.donga.com/news/Economy/article/all...,{'content': '크게보기\n고덕 자연앤 하우스디 투시도. 대보건설 제공\n경...
4,https://www.donga.com/news/Economy/article/all...,{'content': '정부 추산… 전국 평균 1억328만원\n서울은 100만∼3억...


In [91]:
dong_a.dropna(inplace=True)

In [92]:
dong_a.isna().sum()

URL        0
content    0
dtype: int64

In [93]:
dong_a['date'] = pd.to_datetime(dong_a["URL"].apply(lambda x: "-".join(x.split("/")[7:8])))

In [94]:

# 문자열을 실제 딕셔너리로 변환
parsed_dict = ast.literal_eval(dong_a['content'][1])

# 'content'의 값을 꺼내고, \n 제거
cleaned_text = parsed_dict['content'].replace('\\n', ' ')

In [95]:

# 2. 'content' 문자열을 실제 딕셔너리로 변환 후 \n 제거
def parse_and_clean(content_str):
    try:
        parsed = ast.literal_eval(content_str)
        return parsed['content'].replace('\\n', ' ').replace('\n', ' ')
    except (ValueError, SyntaxError, KeyError):
        return None  # 오류가 있을 경우 None 반환
dong_a['content'] = dong_a['content'].apply(parse_and_clean)

In [96]:
dong_a.drop((['URL']), inplace=True, axis=1)


In [97]:
dong_a = dong_a[['date','content']]

In [98]:
dong_a.sort_values(by='date', ascending=False, inplace=True)

In [99]:
dong_a = dong_a.reset_index(drop=True)

In [101]:
dong_a.head()

,date,content
0,2025-06-15,서울시내 아파트의 모습. 2025.5.26/뉴스1 ⓒ News1 지난달 서울 ‘강남...
1,2025-06-15,서울 강남구 티몬 본사 . 2024.7.29/뉴스1 ⓒ News1 이커머스 티몬 인...
2,2025-06-15,"5월 서울 아파트 거래량 증가, 신고가 경신 단지 속출 “아파트 주민들 얘기를 들어..."
3,2025-06-14,서울 집값 9개월 만에 최대폭 상승…시장 과열 ‘경고음’ 강남 3구·마용성 주도…정...
4,2025-06-14,[돈의 심리] 투자로 연 10% 이상 고수익 올릴 능력 있기 때문 투자로 큰돈을 번...


In [3]:
interim_dong_a = pd.read_csv('../data/interim/news/interim_dong_a.csv')

In [5]:
interim_dong_a

,date,content
0,2025-06-15,"5월 서울 아파트 거래량 증가, 신고가 경신 단지 속출 “아파트 주민들 얘기를 들어..."
1,2025-06-15,서울 강남구 티몬 본사 . 2024.7.29/뉴스1 ⓒ News1 이커머스 티몬 인...
2,2025-06-15,서울시내 아파트의 모습. 2025.5.26/뉴스1 ⓒ News1 지난달 서울 ‘강남...
3,2025-06-15,"사진은 SK브로드밴드 가산 AI 데이터센터(AIDC) 모습. (자료사진, SKT 제..."
4,2025-06-14,서울 집값 9개월 만에 최대폭 상승…시장 과열 ‘경고음’ 강남 3구·마용성 주도…정...
...,...,...
6915,2022-12-16,이번 주 서울의 아파트 매매가격이 올해 들어 주간 기준으로 가장 많이 하락한 것으로...
6916,2022-12-16,금융당국이 부동산 시장의 연착륙을 위해 다주택자와 임대사업자에 대한 주택담보대출을 ...
6917,2022-12-16,가수 이효리. 뉴스1 가수 이효리가 37억 원대 신당동 일대 신축 빌딩을 현금 매입...
6918,2022-12-16,서울 남산에서 바라본 아파트 단지. 2022.12.8 뉴스1 고금리 고물가 등에 ...
